# Casa 3. Análisis de fallos

**Duración orientativa: 45 min**

Objetivo: con las métricas y las trazas, clasificar las 10 peores consultas en fallos de recuperación,
de contexto o de generación, y probar una mejora sobre ellas.

- **Recuperación**: la página correcta no está en el top-5.
- **Contexto**: la página está, pero el judge dice que el contexto no basta para responder.
- **Generación**: el contexto basta, pero la respuesta no es fiel o no contesta.

In [1]:
from rag_lab import CFG, set_seeds, summary

# CFG.FAST_MODE = False  # corpus de control completo (por defecto se lee FAST_MODE de .env)
# CFG.DOC_PATH = CFG.DATA_DIR / "mi_documento.pdf"  # cualquier PDF sirve
set_seeds()
summary()

Documento:   /home/mikel/github/2627/iab4/reto-1/semana2-reorganizada/data/documento.pdf (existe)
FAST_MODE:   True
Dispositivo: cuda
GEN_MODEL:   google/gemini-3.1-flash-lite
JUDGE_MODEL: openai/gpt-5-mini
VLM_MODEL:   google/gemini-3.1-flash-lite
OpenRouter:  clave configurada


Trazas:      jsonl /home/mikel/github/2627/iab4/reto-1/semana2-reorganizada/trazas/trazas.jsonl  (alumno: profesor)


/home/mikel/github/2627/iab4/reto-1/semana2-reorganizada/rag_lab/tracing.py:49: UserWarning: No se pudo conectar con Langfuse ([Errno 111] Connection refused): las trazas se guardan en JSONL.
  warnings.warn(f"No se pudo conectar con Langfuse ({e}): las trazas se guardan en JSONL.")


In [2]:
import pandas as pd
from rag_lab import evaluation, multimodal, versions

questions = evaluation.load_questions()
pages = versions.pdf.clean_pages()
enriched = versions.enrich(versions.pdf_fragments("cabeceras"), versions.document_title(pages))
rows = []
for name, rag in [("V3", versions.build("V3")), ("V6", versions.build("V6")),
                    ("V7", multimodal.Multimodal(enriched, multimodal.describe_pages(), "siglip"))]:
    d = evaluation.evaluate_rag(rag, questions, register=False)["detalle"]
    d["version"] = name
    rows.append(d)
detail = pd.concat(rows, ignore_index=True)
detail["posición"] = [next((i for i, u in enumerate(r, 1) if set(u) & set(c)), None)
                       for r, c in zip(detail["recuperadas"], detail["correctas"])]

In [3]:
def classify(row):
    if row["posición"] is None or pd.isna(row["posición"]) or row["posición"] > 5:
        return "recuperación"
    if row["relevancia_contexto"] == 0:
        return "contexto"
    if row["fidelidad"] == 0 or row["relevancia_respuesta"] == 0:
        return "generación"
    return "sin fallo"

detail["fallo"] = detail.apply(classify, axis=1)
detail["gravedad"] = (detail["posición"].fillna(11) / 10 + (3 - detail[["fidelidad", "relevancia_respuesta",
                                                                            "relevancia_contexto"]].sum(axis=1)))
worst = detail.sort_values("gravedad", ascending=False).head(10)
print(detail.groupby(["version", "fallo"]).size().unstack(fill_value=0))
worst[["version", "id", "tipo", "posición", "fallo", "pregunta", "respuesta"]]

fallo    contexto  generación  recuperación  sin fallo
version                                               
V3              4           0             0         16
V6              2           1             2         15
V7              3           0             0         17


,version,id,tipo,posición,fallo,pregunta,respuesta
38,V6,q19,visual,NaN,recuperación,"En el diagrama de infinito de MLOps, ¿qué pala...",No aparece en el documento.
34,V6,q15,visual,NaN,recuperación,"En el diagrama de infinito de MLOps, ¿qué pala...",No aparece en el documento.
18,V3,q19,visual,3.0,contexto,"En el diagrama de infinito de MLOps, ¿qué pala...",No aparece en el documento.
14,V3,q15,visual,3.0,contexto,"En el diagrama de infinito de MLOps, ¿qué pala...",No aparece en el documento.
16,V3,q17,visual,2.0,contexto,"En el cronograma de actividades, ¿qué celdas e...",No aparece en el documento.
37,V6,q18,visual,1.0,contexto,"En el cronograma de desglose por trimestre, ¿q...",No aparece en el documento.
58,V7,q19,visual,1.0,contexto,"En el diagrama de infinito de MLOps, ¿qué pala...",No aparece en el documento.
54,V7,q15,visual,1.0,contexto,"En el diagrama de infinito de MLOps, ¿qué pala...",No aparece en el documento.
17,V3,q18,visual,1.0,contexto,"En el cronograma de desglose por trimestre, ¿q...",No aparece en el documento.
57,V7,q18,visual,2.0,contexto,"En el cronograma de desglose por trimestre, ¿q...",No aparece en el documento.


Revisa a mano cada una de las 10: la clasificación automática se basa en el judge y en la página de referencia,
y ambos pueden estar mal (la pregunta del borrador puede ser incorrecta). Abre en Langfuse la traza de alguna
con su `trace_id` y mira qué fragmentos recuperó.

In [4]:
worst[["version", "id", "fallo", "trace_id"]]

,version,id,fallo,trace_id
38,V6,q19,recuperación,ec57a91726a34216a6c642dc64f9ad49
34,V6,q15,recuperación,2f13bb74845c4b80946d0ff212b2ae8e
18,V3,q19,contexto,c404d7deb183453084800179c7d982a2
14,V3,q15,contexto,4782ca97de2b4b719d476b7628c8b6bf
16,V3,q17,contexto,02216748f9c744259fd0c57e3048c1c9
37,V6,q18,contexto,47d3bd6e9b864a268e5bd5979e2a4f9e
58,V7,q19,contexto,1d568f94d8e54d1c8aa66e923ff017c3
54,V7,q15,contexto,f9c5932cbe6c4541b67fada59e081eef
17,V3,q18,contexto,e28d3260665e4b0292a036c750e4912c
57,V7,q18,contexto,95966e202d6041a08f43079c90519014


## Una mejora: dar al generador la imagen de la página

Muchos fallos visuales son de generación o de contexto: la página correcta se recupera, pero la descripción no
contiene el detalle. Probamos a responder pasando al VLM la imagen de la primera página recuperada.

In [5]:
from rag_lab import llm, pdf

images = pdf.render_pages()
trials = []
for _, row in worst.iterrows():
    if not row["recuperadas"]:
        continue
    page = row["recuperadas"][0][0]
    r = llm.vision(f"Responde en español, breve, a partir de la imagen. Si no está, di 'No aparece en el documento.'\n\n"
                f"PREGUNTA: {row['pregunta']}", [images[page]], max_tokens=400, step="mejora_imagen")
    trials.append({"version": row["version"], "id": row["id"], "fallo": row["fallo"], "página usada": page,
                    "respuesta original": row["respuesta"][:150], "respuesta con imagen": r.text[:150]})
pd.DataFrame(trials)

,version,id,fallo,página usada,respuesta original,respuesta con imagen
0,V6,q19,recuperación,6,No aparece en el documento.,No aparece en el documento.
1,V6,q15,recuperación,6,No aparece en el documento.,No aparece en el documento.
2,V3,q19,contexto,6,No aparece en el documento.,No aparece en el documento.
3,V3,q15,contexto,6,No aparece en el documento.,No aparece en el documento.
4,V3,q17,contexto,12,No aparece en el documento.,No aparece en el documento. El cronograma util...
5,V6,q18,contexto,20,No aparece en el documento.,"En la fila correspondiente a la ""VISITA DE TRA..."
6,V7,q19,contexto,1,No aparece en el documento.,La palabra que aparece en la parte inferior de...
7,V7,q15,contexto,1,No aparece en el documento.,"La palabra que aparece es ""TEST &""."
8,V3,q18,contexto,20,No aparece en el documento.,"En la fila correspondiente a la ""VISITA DE TRA..."
9,V7,q18,contexto,13,No aparece en el documento.,No aparece en el documento.


Prueba: compara las respuestas con la imagen de la página. ¿En cuántas mejora? ¿Qué coste por pregunta añadiría
usar el VLM como generador solo cuando la primera unidad recuperada es una página visual?